# Georeference a IIIF map with Allmaps

Turns IIIF image into a **[Georeference Annotation](https://iiif.io/api/extension/georef/)** that Allmaps can warp onto a web map.

### Mechanism

N pairs are supplied: a pixel `(x, y)` on the scan, and the `(lon, lat)` that pixel sits at in the real world. Allmaps fits a function `f: pixels -> lon/lat` from those pairs, triangulates the map mask into a mesh, pushes each vertex through `f`, and lets the GPU paint IIIF tiles onto the warped triangles. 

Nothing is resampled to disk. The warp is recomputed per frame.

We use the thin plate spline transformation: the surface that passes exactly through every
control point while minimising bending energy. Two consequences:

- Residual at each control point is zero by construction, so a wrong point cannot be spotted by its own error. It silently drags its neighbourhood.
- It interpolates well but extrapolates terribly. Spread the points to the edges of the map.

### The workflow

| Step | Action |
|---|---|
| 1 | Choose a IIIF image |
| 2 | Read pixel positions of towns off the scan |
| 3 | Look up modern coordinates |
| 4 | Trace the map's neat line as a mask |
| 5 | Build `annotation.json` |
| 6 | Check for bad control points |
| 7 | Write the viewer and run it |

## Setup

In [ ]:
%pip install -q requests numpy matplotlib pillow

In [ ]:
import json, math, time, io
from pathlib import Path

import requests
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

USER_AGENT = "iiif-georeference-notebook/0.1"
plt.rcParams["figure.dpi"] = 110

## IIIF image

Eg.
- `https://iiif.micr.io/vVYKG` &mdash; Le Comte de Zeelande, Mortier, c. 1690
- `https://iiif.micr.io/BOjyb` &mdash; Kaarte van Zeeland, Horenbout 1540 / Schroots 1663

In [ ]:
IIIF_ID = "https://iiif.micr.io/BOjyb"
OUT_DIR = Path("demo-zeeland")   # target directory

info = requests.get(f"{IIIF_ID}/info.json", headers={"User-Agent": USER_AGENT}, timeout=30).json()
W, H = info["width"], info["height"]
SERVICE_TYPE = info.get("type", "ImageService3")

print(f"{IIIF_ID}\n{W} x {H} px   {SERVICE_TYPE}   profile={info.get('profile')}")

# Allmaps warps tiles in the browser, so the image server must allow cross-origin reads.
cors = requests.head(f"{IIIF_ID}/info.json", timeout=30).headers.get("access-control-allow-origin")
print("CORS:", cors if cors else "MISSING CORS! The browser will refuse to render this image")

## Step 2 &mdash; Measure pixel positions

`show(x, y, w, h)` fetches that region and plots it **with the axes in original image pixels**, so
you read a town's coordinates straight off the axis ticks, or hover with the cursor.

Start with the whole sheet to find your bearings, then zoom into regions.

In [ ]:
def fetch(x, y, w, h, out_w=1100):
    '''Fetch a IIIF region. Returns (PIL image, region box in source pixels).'''
    x, y = max(0, int(x)), max(0, int(y))
    w, h = min(int(w), W - x), min(int(h), H - y)
    # IIIF servers reject an output wider than the region itself, so never upscale
    out_w = min(int(out_w), w)
    url = f"{IIIF_ID}/{x},{y},{w},{h}/{out_w},/0/default.jpg"
    r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=60)
    r.raise_for_status()
    return Image.open(io.BytesIO(r.content)), (x, y, w, h)


def show(x=0, y=0, w=None, h=None, out_w=1100, grid=True, figsize=(13, 10)):
    '''Plot a region with axes in SOURCE PIXEL coordinates -- read GCP positions off the axes.'''
    w = w if w is not None else W
    h = h if h is not None else H
    img, (x, y, w, h) = fetch(x, y, w, h, out_w)

    fig, ax = plt.subplots(figsize=figsize)
    # extent maps the displayed crop back onto source-pixel coordinates
    ax.imshow(img, extent=[x, x + w, y + h, y])
    ax.set_xlabel("image x (px)")
    ax.set_ylabel("image y (px)")
    ax.set_title(f"region {x},{y},{w},{h}")
    if grid:
        ax.grid(color="cyan", alpha=0.45, linewidth=0.6)
    plt.tight_layout()
    plt.show()
    return ax


show()   # whole sheet

Now zoom in. Adjust the numbers and re-run until the town names are readable, then note the
`x, y` of each town symbol.

**Anchoring rule:** for a walled-town vignette the engraving is drawn *around* the real place, so
take the **centre of the vignette**. For a plain church symbol take its base. Be consistent &mdash;
inconsistency here is a direct error in the fit.

In [ ]:
show(x=1200, y=200, w=1500, h=1000)   # Walcheren, Schouwen and Noord-Beveland

### Record the points

Append as you go. `px, py` come from the plots above; leave `lon, lat` empty &mdash; step 3 fills them.

Pick **12&ndash;20 points spread to the edges**, not clustered in the middle. Only use places that
still exist and have not moved.

In [ ]:
# name -> (px, py)
# Keys are MODERN names so the geocoder finds them; the map's own spelling is in
# the comment. Measured off this scan -- re-check any that look off in step 6.
GCPS = {
    "Brouwershaven":  (2400,  350),
    "Zierikzee":      (2365,  529),   # Zirickzee
    "Steenbergen":    (3217,  618),
    "Veere":          (1638,  633),   # Ter Veer
    "Kortgene":       (2238,  762),   # Cortgeen
    "Arnemuiden":     (1650,  904),   # Armuyen
    "Bergen op Zoom": (3212,  941),
    "Middelburg":     (1567, 1013),   # Middelborch
    "Goes":           (2256, 1028),   # Ter Goes
    "Tholen":         (2625, 1111),
    "Vlissingen":     (1581, 1123),   # Vlissinge
    "Zandvliet":      (3222, 1387),   # Santvliet
    "Oostburg":       (1319, 1558),   # Oostborch
    "Sluis":          ( 999, 1658),   # Sluys
    "Aardenburg":     (1349, 1763),   # Ardenborch
    "Damme":          ( 733, 1825),
    "Brugge":         ( 592, 1874),
    "Maldegem":       ( 983, 1978),
    "Antwerpen":      (3427, 2004),
    "Eeklo":          (1543, 2058),   # Eeckeloo
}

print(f"{len(GCPS)} points recorded")
for n, (px, py) in GCPS.items():
    print(f"  {n:<22} {px:>6}, {py:>6}")

## Step 3 &mdash; Look up modern coordinates

Nominatim covers both the Netherlands and Belgium, which this map needs.

**Read the `kind` column.** `place/...` is a settlement node, which is ideal. `boundary/...` is a
*municipality centroid* &mdash; usually the same spot, but where a modern municipality swallowed
several villages the centroid drifts far from the historic town. Tholen is ~11 km out this way,
Sluis ~9 km, Veere ~6 km.

You do not have to catch these by eye: step 6 flags them. Override any bad one by hand in the next
cell.

In [ ]:
def geocode(name, countrycodes="nl,be", viewbox=None):
    '''Look up a place. Returns (lon, lat, kind) or None.'''
    params = {"format": "jsonv2", "limit": 5, "q": name, "countrycodes": countrycodes}
    if viewbox:                       # (min_lon, min_lat, max_lon, max_lat)
        params["viewbox"] = ",".join(map(str, viewbox))
        params["bounded"] = 1
    r = requests.get("https://nominatim.openstreetmap.org/search",
                     params=params, headers={"User-Agent": USER_AGENT}, timeout=30)
    hits = r.json()
    if not hits:
        return None
    # prefer an actual settlement node over a municipality boundary
    rank = {"city": 0, "town": 1, "village": 2, "hamlet": 3}
    hits.sort(key=lambda h: (h["category"] != "place", rank.get(h["type"], 8)))
    h = hits[0]
    return float(h["lon"]), float(h["lat"]), f"{h['category']}/{h['type']}"


# Restrict to the area the map covers so we don't match a same-named place elsewhere.
VIEWBOX = (2.9, 50.8, 4.9, 52.0)      # Zeeland + Flanders

COORDS = {}
for name in GCPS:
    hit = geocode(name, viewbox=VIEWBOX)
    if hit is None:
        print(f"  MISS      {name}")
    else:
        lon, lat, kind = hit
        COORDS[name] = (lon, lat)
        flag = "  <- municipality centroid" if kind.startswith("boundary") else ""
        print(f"  {name:<22} {lon:9.5f} {lat:9.5f}  {kind}{flag}")
    time.sleep(1.1)                   # Nominatim asks for <=1 request/second

In [ ]:
# Override anything the lookup got wrong, or add places it could not find:
# COORDS["Tholen"] = (4.22328, 51.53545)

MISSING = [n for n in GCPS if n not in COORDS]
print("still missing:", MISSING if MISSING else "none")

## Step 4 &mdash; Mask

The polygon of the *map area*, in image pixels &mdash; the neat line, excluding the printed border,
margins and page. Anything outside it is not drawn.

Keep title and scale cartouches **inside** the mask if they sit within the map frame; cutting them
out would punch holes in the overlay.

Use the corner plots to read the four corners, then set them below.

In [ ]:
# Look at each corner to find the neat line.
for cx, cy in [(0, 0), (W - 600, 0), (W - 600, H - 600), (0, H - 600)]:
    show(cx, cy, 600, 600, out_w=500, figsize=(5, 5))

In [ ]:
# Clockwise from top-left, in image pixels. Default = whole image.
# Traced from the corner crops above: the neat line of this sheet, inset a few px.
MASK = [
    [335,  260],
    [3488, 260],
    [3488, 2592],
    [335,  2592],
]

show(grid=False, figsize=(9, 7))
plt.close()
print("mask:", MASK)

## Step 5 &mdash; Build the annotation

This is the whole Allmaps contract: the IIIF image, the mask, and the control points.

In [ ]:
TRANSFORMATION = "thinPlateSpline"   # or: polynomial, polynomial2, helmert, projective

def build_annotation():
    names = [n for n in GCPS if n in COORDS]
    if len(names) < 3:
        raise ValueError(f"need at least 3 control points, have {len(names)}")

    points = " ".join(f"{int(x)},{int(y)}" for x, y in MASK)
    return {
        "@context": [
            "http://iiif.io/api/extension/georef/1/context.json",
            "http://iiif.io/api/presentation/3/context.json",
        ],
        "type": "Annotation",
        "motivation": "georeferencing",
        "target": {
            "type": "SpecificResource",
            "source": {"id": IIIF_ID, "type": SERVICE_TYPE, "width": W, "height": H},
            "selector": {
                "type": "SvgSelector",
                "value": f'<svg width="{W}" height="{H}"><polygon points="{points}" /></svg>',
            },
        },
        "body": {
            "type": "FeatureCollection",
            "transformation": {"type": TRANSFORMATION},
            "features": [
                {
                    "type": "Feature",
                    "properties": {"resourceCoords": list(GCPS[n]), "label": n},
                    "geometry": {"type": "Point", "coordinates": list(COORDS[n])},
                }
                for n in names
            ],
        },
    }


annotation = build_annotation()
print(f"{len(annotation['body']['features'])} control points, {TRANSFORMATION}")

## Step 6 &mdash; Find bad control points

A thin plate spline fits every control point *exactly*, so residuals are all zero and tell you
nothing. Instead: **leave one out.** Drop each point, fit on the rest, predict the one you dropped,
and measure how far off the prediction is.

One caveat, and it matters: for a point on the **edge** of your control-point cloud, removing it
forces the spline to *extrapolate*, which it does badly. Edge points therefore always score poorly
whether or not they are wrong. The check below marks them `edge` and excludes them from the median,
so only the `INTERIOR` outliers are real suspects.

Judge against the map's own accuracy: on a 16th/17th-century chart several km of error is normal.

In [ ]:
def _tps_solve(src, dst):
    n = len(src)
    d = np.linalg.norm(src[:, None, :] - src[None, :, :], axis=-1)
    K = np.where(d == 0, 0.0, d**2 * np.log(np.where(d == 0, 1.0, d)))
    P = np.hstack([np.ones((n, 1)), src])
    L = np.block([[K, P], [P.T, np.zeros((3, 3))]])
    Y = np.vstack([dst, np.zeros((3, 2))])
    return np.linalg.lstsq(L, Y, rcond=None)[0]


def _tps_apply(coef, src, pts):
    d = np.linalg.norm(pts[:, None, :] - src[None, :, :], axis=-1)
    U = np.where(d == 0, 0.0, d**2 * np.log(np.where(d == 0, 1.0, d)))
    P = np.hstack([np.ones((len(pts), 1)), pts])
    return U @ coef[: len(src)] + P @ coef[len(src):]


def km(a, b):
    (lon1, lat1), (lon2, lat2) = a, b
    dx = (lon2 - lon1) * math.cos(math.radians((lat1 + lat2) / 2)) * 111.32
    return math.hypot(dx, (lat2 - lat1) * 110.57)


names = [n for n in GCPS if n in COORDS]
src = np.array([GCPS[n] for n in names], float)
dst = np.array([COORDS[n] for n in names], float)

def _hull(pts):
    '''Indices of points on the convex hull (Andrew monotone chain).'''
    order = sorted(range(len(pts)), key=lambda i: (pts[i][0], pts[i][1]))
    cross = lambda o, a, b: ((pts[a][0]-pts[o][0])*(pts[b][1]-pts[o][1])
                             - (pts[a][1]-pts[o][1])*(pts[b][0]-pts[o][0]))
    chain = []
    for seq in (order, order[::-1]):
        half = []
        for i in seq:
            while len(half) >= 2 and cross(half[-2], half[-1], i) <= 0:
                half.pop()
            half.append(i)
        chain += half[:-1]
    return set(chain)


edge = _hull(src)

errors = []
for i in range(len(names)):
    keep = [j for j in range(len(names)) if j != i]
    coef = _tps_solve(src[keep], dst[keep])
    pred = _tps_apply(coef, src[keep], src[i:i+1])[0]
    errors.append((km(tuple(dst[i]), tuple(pred)), names[i], i in edge))

errors.sort(reverse=True)
inner = [e for e, _, is_edge in errors if not is_edge]
median = np.median(inner) if inner else 0
print(f"leave-one-out error   (median of interior points: {median:.1f} km)")
print("edge points extrapolate, so their score is inflated -- ignore it\n")
for e, n, is_edge in errors:
    if is_edge:
        tag = "  edge"
    elif e > max(3 * median, 5):
        tag = "  <-- INTERIOR OUTLIER, re-check this one"
    else:
        tag = ""
    print(f"  {n:<22} {e:7.2f} km{tag}")

## Step 7 &mdash; Write the viewer and run it

Writes `annotation.json` and a self-contained `index.html` (MapLibre + `@allmaps/maplibre` from CDN,
no build step) into `OUT_DIR`.

In [ ]:
HTML = """<!doctype html>
<html lang="en">
  <head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1" />
    <title>__TITLE__</title>
    <link rel="stylesheet" href="https://unpkg.com/maplibre-gl@5.24.0/dist/maplibre-gl.css" />
    <style>
      html, body { margin: 0; height: 100%; font-family: system-ui, sans-serif; }
      #map { position: absolute; inset: 0; }
      #panel { position: absolute; top: 1rem; left: 1rem; z-index: 1; width: 16rem;
               padding: .9rem 1rem; border-radius: .5rem; background: rgba(255,255,255,.94);
               box-shadow: 0 1px 4px rgba(0,0,0,.3); font-size: .8rem; line-height: 1.45; }
      #panel h1 { margin: 0 0 .6rem; font-size: .95rem; }
      #opacity { width: 100%; }
      #status { color: #777; } #status[data-state=error] { color: #b00; }
    </style>
  </head>
  <body>
    <div id="map"></div>
    <div id="panel">
      <h1>__TITLE__</h1>
      <label><input type="checkbox" id="visible" checked /> Show historic map</label>
      <label>Opacity <span id="opacity-value">100%</span>
        <input type="range" id="opacity" min="0" max="100" value="100" /></label>
      <div id="status">loading&hellip;</div>
    </div>
    <script type="module">
      import maplibregl from 'https://esm.run/maplibre-gl@5.24.0'
      import { WarpedMapLayer } from 'https://esm.run/@allmaps/maplibre@1.0.0-beta.43'

      const status = document.getElementById('status')
      const map = new maplibregl.Map({
        container: 'map',
        style: 'https://basemaps.cartocdn.com/gl/voyager-gl-style/style.json',
        center: [3.9, 51.4], zoom: 7.5,
        maxPitch: 0,   // the Allmaps plugin does not support pitch
        attributionControl: { customAttribution: 'Map image: Rijksmuseum (CC0)' }
      })
      map.addControl(new maplibregl.NavigationControl({ showCompass: false }))

      const layer = new WarpedMapLayer()
      map.on('load', async () => {
        map.addLayer(layer)
        try {
          const annotation = await fetch('./annotation.json').then((r) => {
            if (!r.ok) throw new Error(`annotation.json: HTTP ${r.status}`)
            return r.json()
          })
          await layer.addGeoreferenceAnnotation(annotation)
          status.textContent =
            `${annotation.body.features.length} control points · ${annotation.body.transformation.type}`
          const bbox = layer.getBbox()
          if (bbox) map.fitBounds(bbox, { padding: 40, duration: 0 })
        } catch (err) {
          // Most likely: opened via file:// so fetch() is blocked. Serve over http.
          status.textContent = err.message
          status.dataset.state = 'error'
        }
      })

      const opacity = document.getElementById('opacity')
      opacity.addEventListener('input', () => {
        layer.setOpacity(Number(opacity.value) / 100)
        document.getElementById('opacity-value').textContent = `${opacity.value}%`
      })
      document.getElementById('visible').addEventListener('change', (e) => {
        layer.setLayerOptions({ visible: e.target.checked })
      })
    </script>
  </body>
</html>
"""

TITLE = "Kaarte van Zeeland (1663) over today"

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "annotation.json").write_text(json.dumps(annotation, indent=2) + "\n")
(OUT_DIR / "index.html").write_text(HTML.replace("__TITLE__", TITLE))

# keep the control points next to the output so the result is reproducible
with open(OUT_DIR / "gcps.csv", "w") as f:
    f.write("name,px,py,lon,lat\n")
    for n in GCPS:
        if n in COORDS:
            f.write(f"{n},{GCPS[n][0]},{GCPS[n][1]},{COORDS[n][0]:.5f},{COORDS[n][1]:.5f}\n")

print("wrote:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p}")
print(f"\nNow serve it (fetch() is blocked on file://):\n"
      f"  python3 -m http.server 8000\n"
      f"  open http://localhost:8000/{OUT_DIR.name}/")

## Iterating

Adjust and re-run steps 5&ndash;7 &mdash; it is cheap:

- **A town sits wrong but its neighbours are fine** &rarr; one bad row. Re-check its pixel position
  and its geocode.
- **A whole region is skewed** &rarr; too few control points there. TPS extrapolates badly, so add
  points nearer the edges.
- **The map looks rubbery** &rarr; set `TRANSFORMATION = "polynomial2"`, or drop outliers.
- **Hunting for a bad point** &rarr; set `TRANSFORMATION = "polynomial"`. Least-squares leaves
  visible residuals, so the worst offender stands out; switch back to `thinPlateSpline` after.

The `annotation.json` is a standards-compliant Georeference Annotation, so it also works in
[Allmaps Viewer](https://viewer.allmaps.org/), in QGIS via
`https://allmaps.xyz/{z}/{x}/{y}.png?url=<public-url>`, and with the Leaflet and OpenLayers plugins.